In [1]:
import warnings
warnings.filterwarnings('ignore')

In [9]:
from datasets import load_dataset

pretraining_dataset= load_dataset(
    "roneneldan/TinyStories",
    split="train"
)

In [10]:
pretraining_dataset = pretraining_dataset.select_columns(
    ["text"]
)

In [11]:
print(pretraining_dataset[0]["text"][:500])

One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them b


In [12]:
instruction_dataset = datasets.load_dataset(
    "c-s-ale/alpaca-gpt4-data",
    split='train'
)
print(instruction_dataset)

README.md: 0.00B [00:00, ?B/s]

data/alpaca_gpt4_data.json:   0%|          | 0.00/43.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 52002
})


In [13]:
i=0
print("Instruction: " + instruction_dataset[i]["instruction"] 
      + "\nInput: " + instruction_dataset[i]["input"] 
      + "\nOutput: " + instruction_dataset[i]["output"])

Instruction: Give three tips for staying healthy.
Input: 
Output: 1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.

2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.

3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.


In [14]:
import os
import requests

In [18]:
code_dir = "./code"

In [19]:
urls = [
    "https://raw.githubusercontent.com/TheAlgorithms/Python/master/searches/double_linear_search_recursion.py",
    "https://raw.githubusercontent.com/KosingZhu/tensorflow/master/tensorflow/python/tools/module_util.py",
    "https://raw.githubusercontent.com/EricRemmerswaal/tensorflow/master/tensorflow/python/distribute/distribute_coordinator_context.py",
    "https://raw.githubusercontent.com/computationalartist/tensorflow/master/tensorflow/python/ops/numpy_ops/integration_test/benchmarks/numpy_mlp.py",
    "https://raw.githubusercontent.com/Van-an/tensorflow/master/tensorflow/python/distribute/coordinator/values.py",
    "https://raw.githubusercontent.com/nkgwer/tensorflow/master/tensorflow/lite/tools/visualize.py",
    "https://raw.githubusercontent.com/gitblazer/youtube-dl/master/youtube_dl/version.py",
    "https://raw.githubusercontent.com/Joshua-Barawa/My-Photos/master/venv/lib/python3.8/site-packages/django/contrib/messages/__init__.py",
    "https://raw.githubusercontent.com/PaliC/pytorch/master/test/fx/test_subgraph_rewriter.py"
]

In [20]:
for url in urls:
    print(f"Working on url: {url}")
    response = requests.get(url)
    file_name = os.path.basename(url)
    file_path = os.path.join(code_dir, file_name)
    
    with open(file_path, "wb") as file:
        file.write(response.content)

Working on url: https://raw.githubusercontent.com/TheAlgorithms/Python/master/searches/double_linear_search_recursion.py
Working on url: https://raw.githubusercontent.com/KosingZhu/tensorflow/master/tensorflow/python/tools/module_util.py
Working on url: https://raw.githubusercontent.com/EricRemmerswaal/tensorflow/master/tensorflow/python/distribute/distribute_coordinator_context.py
Working on url: https://raw.githubusercontent.com/computationalartist/tensorflow/master/tensorflow/python/ops/numpy_ops/integration_test/benchmarks/numpy_mlp.py
Working on url: https://raw.githubusercontent.com/Van-an/tensorflow/master/tensorflow/python/distribute/coordinator/values.py
Working on url: https://raw.githubusercontent.com/nkgwer/tensorflow/master/tensorflow/lite/tools/visualize.py
Working on url: https://raw.githubusercontent.com/gitblazer/youtube-dl/master/youtube_dl/version.py
Working on url: https://raw.githubusercontent.com/Joshua-Barawa/My-Photos/master/venv/lib/python3.8/site-packages/djan

In [21]:
files = os.listdir(code_dir)
for fileA in files:
    print(file)

<_io.BufferedWriter name='C:\\Users\\arsha\\OneDrive\\Desktop\\ai-systems-playground\\pretraining_llms\\code\\test_subgraph_rewriter.py'>
<_io.BufferedWriter name='C:\\Users\\arsha\\OneDrive\\Desktop\\ai-systems-playground\\pretraining_llms\\code\\test_subgraph_rewriter.py'>
<_io.BufferedWriter name='C:\\Users\\arsha\\OneDrive\\Desktop\\ai-systems-playground\\pretraining_llms\\code\\test_subgraph_rewriter.py'>
<_io.BufferedWriter name='C:\\Users\\arsha\\OneDrive\\Desktop\\ai-systems-playground\\pretraining_llms\\code\\test_subgraph_rewriter.py'>
<_io.BufferedWriter name='C:\\Users\\arsha\\OneDrive\\Desktop\\ai-systems-playground\\pretraining_llms\\code\\test_subgraph_rewriter.py'>
<_io.BufferedWriter name='C:\\Users\\arsha\\OneDrive\\Desktop\\ai-systems-playground\\pretraining_llms\\code\\test_subgraph_rewriter.py'>
<_io.BufferedWriter name='C:\\Users\\arsha\\OneDrive\\Desktop\\ai-systems-playground\\pretraining_llms\\code\\test_subgraph_rewriter.py'>
<_io.BufferedWriter name='C:\\User

In [22]:
code_dataset = []
for file in os.listdir(code_dir):
    code_dataset.append(
        {'text': open(os.path.join(code_dir, file), 'r').read()}
    )

In [23]:
code_dataset = datasets.Dataset.from_list(code_dataset)
print(code_dataset)

Dataset({
    features: ['text'],
    num_rows: 9
})


In [24]:
dataset = datasets.concatenate_datasets(
    [pretraining_dataset, code_dataset]
)
print(dataset)

Dataset({
    features: ['text'],
    num_rows: 2119728
})


## Data Cleaning

In [25]:
dataset.num_rows

2119728

In [26]:
import heapq

def paragraph_length_filter(x):
    """Returns False iff a page has too few lines or lines are too short."""
    lines = x['text'].split('\n')
    if (
        len(lines) < 3
        or min(heapq.nlargest(3, [len(line) for line in lines])) < 3
    ):
        return False
    return True

In [27]:
dataset = dataset.filter(
    paragraph_length_filter,
    load_from_cache_file=False
)

Filter:   0%|          | 0/2119728 [00:00<?, ? examples/s]

In [28]:
dataset.num_rows

2000809

In [29]:
def find_duplicates(paragraphs):
    """
    Use this function to find the number of repetitions 
    in the paragraphs.
    """
    unique_x = set()
    duplicate_chars = 0
    duplicate_elements = 0
    for element in paragraphs:
        if element in unique_x:
            duplicate_chars += len(element)
            duplicate_elements += 1
        else:
            unique_x.add(element)
    return duplicate_elements, duplicate_chars

In [31]:
import re

def paragraph_repetition_filter(x):
    """
    Returns False iff a page has too many repetitions.
    """
    text = x['text']
    paragraphs = re.compile(r"\n{2,}").split(text.strip())                # Spliting paragraphs (2 or more newlines)
    paragraphs_duplicates, char_duplicates = find_duplicates(paragraphs)  # Finding number of duplicates in paragraphs
    if paragraphs_duplicates / len(paragraphs) > 0.3:
        return False
    if char_duplicates / len(text) > 0.2:
        return False
    return True

In [32]:
dataset = dataset.filter(
    paragraph_repetition_filter,
    load_from_cache_file=False
)

Filter:   0%|          | 0/2000809 [00:00<?, ? examples/s]

In [33]:
dataset.num_rows

2000804

In [34]:
def deduplication(ds):
    def dedup_func(x):
        """Use this function to remove duplicate entries"""
        if x['text'] in unique_text:
            return False
        else:
            unique_text.add(x['text'])
            return True

    unique_text = set()

    ds = ds.filter(dedup_func, load_from_cache_file=False, num_proc=1)
    return ds

dataset = deduplication(dataset)

Filter (num_proc=1):   0%|          | 0/2000804 [00:00<?, ? examples/s]

In [35]:
dataset.num_rows

1696948

In [41]:
import fasttext

model = fasttext.load_model("lid.176.bin")


def english_language_filter(ds):

    def is_english(x):

        text = x['text'].replace("\n", "")

        language, score = model.predict(text)

        language = language[0].replace("__label__", "")

        return language == "en" and score[0] > 0.4

    return ds.filter(
        is_english,
        load_from_cache_file=False
    )


dataset = english_language_filter(dataset)

Filter:   0%|          | 0/1696948 [00:00<?, ? examples/s]

In [42]:
dataset.num_rows

1696946

In [46]:
file_path = "./data/dataset.parquet"

dataset.to_parquet(file_path)

Creating parquet from Arrow format:   0%|          | 0/16 [00:00<?, ?ba/s]

1555035297